In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import spearmanr, zscore
from statsmodels.stats.multitest import multipletests

In [ ]:
expr = pd.read_csv(
    "data/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    index_col=0
)

models = pd.read_csv("data/Model.csv")

prism = pd.read_csv(
    "data/secondary-screen-dose-response-curve-parameters.csv"
)

treatments = pd.read_csv(
    "data/secondary-screen-replicate-collapsed-treatment-info.csv"
)

cell_info = pd.read_csv(
    "data/secondary-screen-cell-line-info.csv"
)

In [ ]:
drug_tests = [
    "palbociclib",
    "ribociclib",
    "abemaciclib",
    "MK-1775",
    "LY2603618",
    "SCH-900776",
    "erastin",
    "sulfasalazine",
]
def gene_symbol(col):
    return str(col).split(" ")[0]


symbol_to_col = {
    gene_symbol(col): col
    for col in expr.columns
}
symbol_to_col

In [ ]:
expr = expr.rename(columns=lambda col: gene_symbol(col))

In [ ]:
expr

In [ ]:
def build_mss_tmbh_score(
    expr,
    model_ids,
    de,
    gene_col="gene",
    lfc_col="log2FoldChange",
    padj_col="padj",
    padj_cutoff=0.05,
    lfc_cutoff=1,
    top_n=50,
):
    """
    Build an MSS/TMB-H transcriptional similarity score.

    Score =
        mean(z-scored upregulated genes)
        -
        mean(z-scored downregulated genes)

    Higher score = more similar to MSS/TMB-H.
    """

    
    sig = de[
        (de[padj_col] < padj_cutoff) &
        (de[lfc_col].abs() >= lfc_cutoff)
    ].copy()

    
    up = (
        sig[sig[lfc_col] > 0]
        .sort_values(lfc_col, ascending=False)
        .head(top_n)
    )

    down = (
        sig[sig[lfc_col] < 0]
        .sort_values(lfc_col, ascending=True)
        .head(top_n)
    )

    up_genes = up[gene_col].astype(str).tolist()
    down_genes = down[gene_col].astype(str).tolist()

    
    up_available = [
        g for g in up_genes
        if g in symbol_to_col
    ]

    down_available = [
        g for g in down_genes
        if g in symbol_to_col
    ]

    print(f"UP genes available: {len(up_available)}/{len(up_genes)}")
    print(f"DOWN genes available: {len(down_available)}/{len(down_genes)}")

    if len(up_available) == 0 or len(down_available) == 0:
        raise ValueError(
            "No usable up- or downregulated genes were found in DepMap."
        )

    
    model_ids = set(model_ids)

    x = expr.loc[
        expr.index.intersection(model_ids),
        [g for g in up_available + down_available]
    ].copy()

    
    x_z = x.apply(
        lambda col: zscore(col, nan_policy="omit"),
        axis=0
    )

    up_cols = [
        g
        for g in up_available
    ]

    down_cols = [
        g
        for g in down_available
    ]

    score = (
        x_z[up_cols].mean(axis=1)
        -
        x_z[down_cols].mean(axis=1)
    )

    score = score.rename("MSS_TMBH_score")

    score_df = score.reset_index()
    score_df.columns = [
        "DepMap_ID",
        "MSS_TMBH_score"
    ]

    return score_df, up_available, down_available

In [ ]:
crc_models = models[
    models["OncotreeCode"] ==
    "COAD"
].copy()

print("CRC models:", len(crc_models))

In [ ]:
ucec_models = models[
    models["OncotreeCode"] == "UCEC"
].copy()

print("Uterus models:", len(ucec_models))

In [ ]:
de_crc = pd.read_csv(
    "data/DE_results/MSS_TMB_H_vs_MSS_TMB_L.csv", 
    index_col=0
)
de_crc["gene"] = de_crc.index.astype(str).str.upper()
de_crc["log2FoldChange"] = de_crc["logFC"].astype(float)
de_crc["padj"] = de_crc["FDR"].astype(float)
de_crc

In [ ]:
de_ucec = pd.read_csv(
    "data/DE_results/ucec/MSS_TMB_H_vs_MSS_TMB_L.csv", 
    index_col=0
)
de_ucec["gene"] = de_ucec.index.astype(str).str.upper()
de_ucec["log2FoldChange"] = de_ucec["logFC"].astype(float)
de_ucec["padj"] = de_ucec["FDR"].astype(float)
de_ucec

In [ ]:
expr.index = expr["ModelID"]
expr

In [ ]:
crc_score, crc_up, crc_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=crc_models["ModelID"],
    de=de_crc,
    top_n=50
)

crc_score.head()

In [ ]:
ucec_score, ucec_up, ucec_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=ucec_models["ModelID"],
    de=de_ucec,
    top_n=50
)

ucec_score.head()

In [ ]:
def test_signature_drug_response(
    score_df,
    prism,
    drugs,
    response_col="auc",
):
    results = []

    for drug in drugs:

        drug_df = prism[
            prism["name"].str.lower() == drug.lower()
        ].copy()

        if drug_df.empty:
            print(f"Drug not found: {drug}")
            continue

        df = score_df.merge(
            drug_df[
                [
                    "depmap_id",
                    "name",
                    response_col
                ]
            ],
            left_on="DepMap_ID",
            right_on="depmap_id",
            how="inner"
        )

        df = df.replace(
            [np.inf, -np.inf],
            np.nan
        )

        df = df.dropna(
            subset=[
                "MSS_TMBH_score",
                response_col
            ]
        )

        if len(df) < 5:
            print(
                f"Skipping {drug}: only {len(df)} models"
            )
            continue

        rho, p = spearmanr(
            df["MSS_TMBH_score"],
            df[response_col]
        )

        results.append({
            "drug": drug,
            "n": len(df),
            "rho": rho,
            "pvalue": p,
        })

    results = pd.DataFrame(results)

    if not results.empty:

        results["FDR"] = multipletests(
            results["pvalue"],
            method="fdr_bh"
        )[1]

        results = results.sort_values(
            "pvalue"
        )

    return results

In [ ]:
crc_results = test_signature_drug_response(
    score_df=crc_score,
    prism=prism,
    drugs=drug_tests
)

crc_results

In [ ]:
ucec_results = test_signature_drug_response(
    score_df=ucec_score,
    prism=prism,
    drugs=drug_tests
)

ucec_results

In [ ]:
crc_results["cancer"] = "CRC"
ucec_results["cancer"] = "UCEC"

all_results = pd.concat(
    [
        crc_results,
        ucec_results
    ],
    ignore_index=True
)

all_results = all_results[
    [
        "cancer",
        "drug",
        "n",
        "rho",
        "pvalue",
        "FDR"
    ]
]

all_results.sort_values(
    ["cancer", "pvalue"]
)